# T Industries - governance objects (run before the pipeline)
Creates the row-level security objects (prefix `gov_`, schema `t_hardware`) that `t_hardware.gold_inventory_position` references in its `WITH ROW FILTER` clause (functions cannot be created inside a pipeline):

* `t_hardware.gov_warehouse_access` - mapping of user / account group -> warehouse (`Delta BC`, `Toronto` or `ALL`)
* `t_hardware.gov_warehouse_row_filter(warehouse)` - returns true only for warehouses mapped to `current_user()` or one of their groups

**Demo tip:** set `presenter_scope` to `Delta BC` and rerun this notebook (or just `UPDATE` the presenter's row) - the same `SELECT * FROM t_hardware.gold_inventory_position` immediately returns only BC rows. Set it back to `ALL` afterwards.

In [0]:
dbutils.widgets.text("catalog", "mdk_demo_catalog")
dbutils.widgets.dropdown("presenter_scope", "ALL", ["ALL", "Delta BC", "Toronto"])
C = dbutils.widgets.get("catalog")
SCOPE = dbutils.widgets.get("presenter_scope")
ME = spark.sql("SELECT current_user()").first()[0]
G = f"{C}.t_hardware"     # single schema; governance objects use the gov_ prefix
print(f"catalog={C}  presenter={ME}  scope={SCOPE}")

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {G}.gov_warehouse_access (
  principal STRING COMMENT 'User email or account group name',
  warehouse STRING COMMENT 'Warehouse the principal may see: Delta BC, Toronto, or ALL')
COMMENT 'Row-level security mapping for gold_inventory_position: which users / groups may see which warehouse. Edit this table to change access - no pipeline rerun needed.'""")
# Regional ops groups (create these account groups to demo with real users) + the presenter
rows = [("t_bc_ops", "Delta BC"), ("t_on_ops", "Toronto"), ("t_exec", "ALL"), (ME, SCOPE)]
spark.createDataFrame(rows, "principal string, warehouse string").createOrReplaceTempView("wa_src")
spark.sql(f"""MERGE INTO {G}.gov_warehouse_access t USING wa_src s ON t.principal = s.principal
  WHEN MATCHED THEN UPDATE SET warehouse = s.warehouse WHEN NOT MATCHED THEN INSERT *""")
display(spark.table(f"{G}.gov_warehouse_access"))

In [0]:
spark.sql(f"""CREATE OR REPLACE FUNCTION {G}.gov_warehouse_row_filter(warehouse STRING)
RETURNS BOOLEAN
COMMENT 'Row filter for gold_inventory_position: true when the warehouse is mapped to the current user or one of their account groups (or ALL) in gov_warehouse_access.'
RETURN EXISTS (
  SELECT 1 FROM {G}.gov_warehouse_access a
  WHERE (a.warehouse = 'ALL' OR a.warehouse = gov_warehouse_row_filter.warehouse)
    AND (a.principal = current_user() OR is_account_group_member(a.principal)))""")
print("row filter ready")

In [0]:
if spark.catalog.tableExists(f"{G}.gold_inventory_position"):
    display(spark.sql(f"SELECT warehouse, count(*) AS skus, sum(on_hand_qty) AS on_hand FROM {G}.gold_inventory_position GROUP BY warehouse"))
else:
    print("gold_inventory_position not built yet - run pipeline t_genie_ready")